<a href="https://colab.research.google.com/github/Randomz4-debug/GoogleCollab-Projects/blob/main/MNISTSIGNLANGUAGECNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [41]:
class Model(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Linear(784, 128)
    self.conv2 = nn.Linear(128, 64)
    self.conv3 = nn.Linear(64, 32)
    self.conv4 = nn.Linear(32, 16)
    self.conv5 = nn.Linear(16, 25) # Changed from 26 to 25 to match unique label count

  def forward(self, x):

    x = self.conv1(x)
    x = F.relu(x)

    x = self.conv2(x)
    x = F.relu(x)

    x = self.conv3(x)
    x = F.relu(x)

    x = self.conv4(x)
    x = F.relu(x)

    x = self.conv5(x)

    return x

In [44]:
model = Model()

In [1]:
import pandas as pd

url = 'https://raw.githubusercontent.com/namas191297/sign_language_mnist_cnn/refs/heads/master/data/sign_mnist_train.csv'

my_df = pd.read_csv(url)

In [2]:
X_train = my_df.drop('label', axis=1)
y_train = my_df['label']

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch

# Re-derive X_train, X_test, y_train, y_test from my_df to ensure a clean state
# This addresses the 'builtin_function_or_method' error by resetting y_train/y_test
# and also helps with the UserWarning for X_train/X_test if they were already tensors.

# Assume my_df is still available in the kernel from previous cells.
X_data_raw = my_df.drop('label', axis=1)
y_data_raw = my_df['label']

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_data_raw, y_data_raw, test_size=0.2, random_state=42
)

# Convert to numpy arrays if they are pandas DataFrames/Series (train_test_split might return Series)
if isinstance(X_train_np, pd.DataFrame):
    X_train_np = X_train_np.to_numpy()
if isinstance(X_test_np, pd.DataFrame):
    X_test_np = X_test_np.to_numpy()
if isinstance(y_train_np, pd.Series):
    y_train_np = y_train_np.to_numpy()
if isinstance(y_test_np, pd.Series):
    y_test_np = y_test_np.to_numpy()

# Apply the scaling for X_train and X_test (as was done in cell IOcF6dfsXGt9)
X_train_np = X_train_np / 255.0
X_test_np = X_test_np / 255.0

# Convert to torch tensors
X_train = torch.tensor(X_train_np, dtype=torch.float32)
X_test = torch.tensor(X_test_np, dtype=torch.float32)

y_train = torch.tensor(y_train_np, dtype=torch.long)
y_test = torch.tensor(y_test_np, dtype=torch.long)

In [45]:
criterion = nn.CrossEntropyLoss()

optim = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 500
losses = []

for epoch in range(epochs):
  y_pred = model(X_train)

  loss = criterion(y_pred, y_train)
  losses.append(loss)

  # These steps should happen in every epoch to update the model
  optim.zero_grad()
  loss.backward()
  optim.step()

  if epoch % 10 == 0:
    print(f'Epoch: {epoch} Loss: {loss.item()}')

Epoch: 0 Loss: 0.5337878465652466
Epoch: 10 Loss: 0.5285592675209045
Epoch: 20 Loss: 0.5246983170509338
Epoch: 30 Loss: 0.6275337338447571
Epoch: 40 Loss: 0.5253251194953918
Epoch: 50 Loss: 0.5205612778663635
Epoch: 60 Loss: 0.510915994644165
Epoch: 70 Loss: 0.5024635195732117
Epoch: 80 Loss: 0.4960307776927948
Epoch: 90 Loss: 0.4912710189819336
Epoch: 100 Loss: 0.4864475429058075
Epoch: 110 Loss: 0.4817703664302826
Epoch: 120 Loss: 0.47740238904953003
Epoch: 130 Loss: 0.48520737886428833
Epoch: 140 Loss: 0.7950133085250854
Epoch: 150 Loss: 0.6826908588409424
Epoch: 160 Loss: 0.5015211701393127
Epoch: 170 Loss: 0.4845389723777771
Epoch: 180 Loss: 0.462991863489151


In [53]:
torch.save(model.state_dict(), "model.pt")

In [54]:

with torch.no_grad():
  y_pred = model.forward(X_test)
  correct = torch.eq(torch.max(F.softmax(y_pred, dim=1), dim=1)[1], y_test).sum().item()
  print(f'Accuracy: {correct / len(y_test)}')

Accuracy: 0.8162447641595337
